# 05 — o200k Tokenizer Measurement (Light Scaffold, Synthetic Only)

이 노트북은 고정된 tiktoken `o200k_base` 구현으로 T_KO/T_EN/TP와 그 정확한 log 가법적 분해를 측정하는 구현을 **synthetic 예시로만** 검증한다. **candidate cohort의 full population tokenization 실행은 이 노트북의 범위 밖이다** — G1/Human audit close 이후 별도로 승인·실행한다.

## 0 Contract / Scope

- Tokenizer: 고정된 `tiktoken==0.13.0`의 `o200k_base`, `.runtime/tiktoken-cache`의 승인된 offline artifact만 사용(network fallback 금지).
- 산출 quantity: `T_KO`, `T_EN`, `TP`, `logTP`, `token_difference`, `CodePointRatio`, `ByteDensityRatio`, `CompressionPenalty`, `roundtrip_ok`.
- Identity: `logTP = logCodePointRatio + logByteDensityRatio + logCompressionPenalty`, epsilon=1e-10.
- 금지: candidate cohort 전체 tokenization 실행, 형태소/regex chunk mechanism, 통계적 추론.

In [1]:
from __future__ import annotations

from tokenization_premium.paths import PROJECT_ROOT
from tokenization_premium.tokenization import load_o200k_base_offline
from tokenization_premium.tokenizer_measurement import (
    EXACT_DECOMPOSITION_EPSILON,
    pair_token_measurement,
)

FORBIDDEN_COMPONENTS = {'full_population_tokenization', 'regex_chunk_mechanism', 'morphology', 'statistical_inference'}
encoding = load_o200k_base_offline(PROJECT_ROOT / '.runtime/tiktoken-cache')
{'encoding_name': encoding.name, 'epsilon': EXACT_DECOMPOSITION_EPSILON}

{'encoding_name': 'o200k_base', 'epsilon': 1e-10}

## 1 Synthetic pair measurement

SSOT tokenizer roundtrip 계약과 동일한 synthetic 예시로 measurement 구현을 검증한다.

In [2]:
SYNTHETIC_PAIRS = [
    ('안녕하세요 반갑습니다', 'Hello nice to meet you'),
    ('한국어 토큰화 재현성 검사', 'Korean tokenization reproducibility check'),
    ('공백  두 개와\n줄바꿈', 'two  spaces and\na line break'),
    ('NFC 한글과 emoji 😀 123', 'NFC Hangul and emoji 😀 123'),
]
measurements = [
    {'ko': ko, 'en': en, **pair_token_measurement(ko, en, encoding=encoding)}
    for ko, en in SYNTHETIC_PAIRS
]
[
    {
        'T_KO': m['T_KO'], 'T_EN': m['T_EN'], 'TP': round(m['TP'], 6),
        'roundtrip_ok': m['roundtrip_ok'], 'identity_abs_error': m['identity_abs_error'],
    }
    for m in measurements
]

[{'T_KO': 5,
  'T_EN': 5,
  'TP': 1.0,
  'roundtrip_ok': True,
  'identity_abs_error': 2.220446049250313e-16},
 {'T_KO': 9,
  'T_EN': 7,
  'TP': 1.285714,
  'roundtrip_ok': True,
  'identity_abs_error': 5.551115123125783e-17},
 {'T_KO': 10,
  'T_EN': 8,
  'TP': 1.25,
  'roundtrip_ok': True,
  'identity_abs_error': 0.0},
 {'T_KO': 9,
  'T_EN': 9,
  'TP': 1.0,
  'roundtrip_ok': True,
  'identity_abs_error': 1.457167719820518e-16}]

## 2 Exact decomposition identity check

모든 synthetic pair에서 `logTP = logCodePointRatio + logByteDensityRatio + logCompressionPenalty`가 epsilon=1e-10 이내로 성립하는지, roundtrip이 전부 성공했는지 확인한다.

In [3]:
for m in measurements:
    reconstructed = m['logCodePointRatio'] + m['logByteDensityRatio'] + m['logCompressionPenalty']
    assert abs(m['logTP'] - reconstructed) < EXACT_DECOMPOSITION_EPSILON
    assert m['roundtrip_ok'] is True
{
    'all_identities_hold': True,
    'all_roundtrips_ok': True,
    'max_identity_abs_error': max(m['identity_abs_error'] for m in measurements),
}

{'all_identities_hold': True,
 'all_roundtrips_ok': True,
 'max_identity_abs_error': 2.220446049250313e-16}

## 3 Next step

구현과 synthetic 검증이 통과했다. Full population token measurement (`TOKEN_MEASUREMENTS_O200K_v001.parquet`, N=3,836,013)은 G1/Human audit close 이후 별도 배정에서 승인·실행한다.